# GrandNexus v7.2.4-rc — the loop that learns, run 5a = recall (pre-registered, logbook 57)
Memory **fixed** at 61 experiences (shipped in the zip, no experience phase). Test = every never-used FOLIO train item whose
story (premises text) is already in the memory: **68 items on 38 stories, sha `4895b2c3fc494d7a`** (yes 26, no 23, unknown 19).
Conditions **none / random / learned**, k = 3, about 200 calls.

**C-recall** (fixed before measuring): paired sign test learned vs none with p < 0.05 in favour of learned, asserted errors not
above none. Secondary: learned > random (the relevance of the same story, not the mere presence of examples).
Expected: learned > none on about 10 items against 0.

Please run the cells exactly as they are: the protocol is the result. Cell 4 checks the frozen set before any call; cell 5 can be
re-run after an interruption (it resumes from its checkpoint). No item text is printed.

## 1. Get GrandNexus (latest zip on the Drive, or upload)

In [ ]:
import os, glob, subprocess, zipfile
DRIVE_DIR = '/content/drive/MyDrive/GrandNexus'
WORK = '/content/GrandNexus'
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e:
    print('Drive not mounted:', e)
zips = sorted(glob.glob(os.path.join(DRIVE_DIR, 'GrandNexus_v7.2.4*.zip')) + glob.glob('/content/drive/MyDrive/GrandNexus_Sessions/GrandNexus_v7.2.4*.zip'),
              key=os.path.getmtime)
if not zips:
    zips = sorted(glob.glob(os.path.join(DRIVE_DIR, 'GrandNexus_*.zip')), key=os.path.getmtime)
if zips:
    source = zips[-1]
else:
    from google.colab import files
    source = '/content/' + next(iter(files.upload()))
if not os.path.exists(WORK):
    z = zipfile.ZipFile(source)
    top = {n.split('/')[0] for n in z.namelist()}
    z.extractall('/content' if top == {'GrandNexus'} else WORK)   # zip with or without the GrandNexus/ top folder
    print('extracted from:', source)
print(subprocess.run('git checkout -q main && git describe --tags && git log --oneline -1 && git branch', shell=True, cwd=WORK, capture_output=True, text=True).stdout)

## 2. Packages, key, pre-flight check (1 call)

In [ ]:
!pip install -q anthropic clingo sympy networkx psutil
import os, anthropic
from google.colab import userdata
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
m = anthropic.Anthropic().messages.create(model='claude-sonnet-5-5', max_tokens=10, messages=[{'role': 'user', 'content': 'Answer only: OK'}])
print('reply:', m.content[0].text, '| id:', m.id, '| model:', m.model)

## 3. Offline tests (0 calls)

In [ ]:
%cd /content/GrandNexus
!python -W ignore tests_lab/test_nexus_logic.py | tail -1
!python -W ignore tests_lab/test_learning_loop.py | tail -1
!python -W ignore tests_lab/test_bench_story.py | tail -1

## 4. Pre-registration check (0 calls): the frozen set must be 68 items, sha 4895b2c3fc494d7a

In [ ]:
import subprocess
WORK = '/content/GrandNexus'
CMD = 'python tools/bench_learning.py --split train --memory tests_lab/resultats/learning/memory_train_61exp_2026-11-04_191343.json --exp-n 0 --story-mode in-memory --test-n 0 --test-seed 7 --k 3 --conditions none,random,learned --exclude-from "tests_lab/resultats/learning/*.json" "tests_lab/resultats/folio/folio_train_*.json"'
out = subprocess.run(CMD + ' --dry-run', shell=True, cwd=WORK, capture_output=True, text=True)
print(out.stdout, out.stderr[-500:])
assert 'test set: 68 items (sha 4895b2c3fc494d7a)' in out.stdout, 'NOT the pre-registered set: stop and report'
print('pre-registered set confirmed')

## 5. Run 5a (about 200 calls; re-run this cell after an interruption, it resumes)

In [ ]:
import subprocess, sys
WORK = '/content/GrandNexus'
CMD = 'python tools/bench_learning.py --split train --memory tests_lab/resultats/learning/memory_train_61exp_2026-11-04_191343.json --exp-n 0 --story-mode in-memory --test-n 0 --test-seed 7 --k 3 --conditions none,random,learned --exclude-from "tests_lab/resultats/learning/*.json" "tests_lab/resultats/folio/folio_train_*.json"'
p = subprocess.Popen(CMD, shell=True, cwd=WORK, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    sys.stdout.write(line)
p.wait(); print('exit code', p.returncode)

## 6. Save to Drive (commit + complete zip)

In [ ]:
import os, glob, subprocess, shutil, time
DRIVE_DIR = '/content/drive/MyDrive/GrandNexus'
WORK = '/content/GrandNexus'
for p in glob.glob(WORK + '/**/__pycache__', recursive=True): shutil.rmtree(p, ignore_errors=True)
print(subprocess.run('git add -A && git -c user.name=Khaled -c user.email=khaled@grandnexus.local commit -qm "learning loop run 5a (recall)"; git log --oneline -1',
                     shell=True, cwd=WORK, capture_output=True, text=True).stdout)
name = os.path.join(DRIVE_DIR, 'GrandNexus_session_run5a_' + time.strftime('%Y%m%d_%H%M'))
os.makedirs(DRIVE_DIR, exist_ok=True); shutil.make_archive(name, 'zip', '/content', 'GrandNexus'); print('saved:', name + '.zip')
print('result files:', sorted(glob.glob(WORK + '/tests_lab/resultats/learning/learning_train_test68_*.json')))